In [ ]:
import os
import re
import json
import time
import requests

from bs4 import BeautifulSoup

from urllib.parse import (
    urljoin,
    urlparse,
    urlunparse
)

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [1]:
#BLOQUE 2 (Constantes)
HEADERS = {

    "User-Agent":
    "Mozilla/5.0 (compatible; UPV-Admision-Bot/1.0)"

}

PAUSA = 0.3


FUENTES = [

    (
        "Bachillerato",
        "https://www.upv.es/admision/admision-grado/bachillerato-es.html"
    ),

    (
        "Ciclos formativos",
        "https://www.upv.es/admision/admision-grado/ciclos-formativos-es.html"
    ),

    (
        "Titulados universitarios",
        "https://www.upv.es/admision/admision-grado/titulados-universitarios-es.html"
    ),

    (
        "Mayores de 25/40/45 años",
        "https://www.upv.es/admision/admision-grado/mayores-25-40-45-es.html"
    ),

    (
        "Vengo de otra universidad",
        "https://www.upv.es/admision/admision-grado/vengo-de-otra-universidad-es.html"
    )

]

In [ ]:
#BLOQUE 3 (Ruta del JSON en directorio en el que estamos)
NOMBRE_PROGRAMA = "Sacar_Admision_Grado.ipynb"


ruta_programa = None

for root, dirs, files in os.walk("/content/drive/MyDrive"):

    if NOMBRE_PROGRAMA in files:

        ruta_programa = root

        break


if ruta_programa is None:

    raise Exception(
        "No se ha encontrado el notebook."
    )


CARPETA_JSON = os.path.join(

    ruta_programa,

    "JSONs"

)

os.makedirs(

    CARPETA_JSON,

    exist_ok=True

)

In [ ]:
#BLOQUE 4 (Funciones auxiliares)
def limpiar_texto(txt):

    if txt is None:

        return ""

    return re.sub(

        r"\s+",

        " ",

        txt

    ).strip()


def normalizar_url(url):

    p = urlparse(url)

    return urlunparse(

        (

            p.scheme,

            p.netloc,

            p.path,

            "",

            "",

            ""

        )

    )


def get(url):

    try:

        r = requests.get(

            url,

            headers=HEADERS,

            timeout=20

        )

        r.raise_for_status()

        time.sleep(PAUSA)

        return BeautifulSoup(

            r.text,

            "html.parser"

        )

    except Exception:

        return None


def texto(tag, selector=None):

    if selector is not None:

        tag = tag.select_one(selector)

    if tag is None:

        return ""

    return limpiar_texto(

        tag.get_text(

            " ",

            strip=True

        )

    )


def url_absoluta(base, href):

    if not href:

        return ""

    return normalizar_url(

        urljoin(

            base,

            href

        )

    )

In [ ]:
# ==========================================================
# BLOQUE 5. Extraer tarjetas de una sección
# ==========================================================

def extraer_tarjetas_seccion(section, url_base):

    tarjetas = []

    slider = section.select_one(
        "div.slider-content"
    )

    if slider is None:
        return tarjetas

    for card in slider.select(
        "div.card-bg"
    ):

        titulo = texto(
            card,
            "h3"
        )

        descripcion = texto(
            card,
            "p.text-sm"
        )

        enlace = card.select_one(
            "a[href]"
        )

        if enlace is None:
            continue

        tarjetas.append(

            {

                "titulo": titulo,

                "descripcion": descripcion,

                "texto_enlace": limpiar_texto(
                    enlace.get_text()
                ),

                "url": url_absoluta(
                    url_base,
                    enlace["href"]
                )

            }

        )

    return tarjetas


In [ ]:
# ==========================================================
# BLOQUE 6. Extraer secciones y contenido
# ==========================================================

def extraer_padre(url):

    print(url)

    soup = get(url)

    if soup is None:
        return None

    titulo = ""

    if soup.title:
        titulo = limpiar_texto(
            soup.title.get_text()
        )

    main = soup.find(
        "main",
        class_="search-page"
    )

    if main is None:
        return None

    pagina = {

        "titulo": titulo,
        "url": normalizar_url(url),
        "secciones": []

    }

    for i in range(1, 7):

        sec = main.find(
            id=f"section-{i:02d}"
        )

        if sec is None:
            continue

        # --------------------------------------------------
        # Información de la sección
        # --------------------------------------------------

        titulo_sec = ""
        descripcion_sec = ""

        info = sec.find(
            "div",
            class_="col-4"
        )

        if info:

            h = info.find(["h2", "h3"])

            if h:
                titulo_sec = limpiar_texto(
                    h.get_text()
                )

            p = info.find(
                "p",
                class_="text-sm"
            )

            if p:
                descripcion_sec = limpiar_texto(
                    p.get_text()
                )

        datos_seccion = {

            "id": f"section-{i:02d}",
            "titulo": titulo_sec,
            "descripcion": descripcion_sec,

            "tarjetas": [],
            "acordeones": []

        }

        # ==================================================
        # TARJETAS (.card-bg)
        # ==================================================

        for card in sec.select(".card-bg"):

            titulo_card = ""
            descripcion_card = ""
            enlace = None

            h = card.find(["h3", "h4"])

            if h:
                titulo_card = limpiar_texto(
                    h.get_text()
                )

            p = card.find(
                "p",
                class_="text-sm"
            )

            if p:
                descripcion_card = limpiar_texto(
                    p.get_text()
                )

            a = card.find(
                "a",
                href=True
            )

            if a:

                enlace = normalizar_url(
                    urljoin(
                        url,
                        a["href"]
                    )
                )

            datos_seccion["tarjetas"].append({

                "titulo": titulo_card,
                "descripcion": descripcion_card,
                "url": enlace

            })

        # ==================================================
        # ACORDEONES
        # ==================================================

        for acc in sec.select(".accordion-element-content"):

            titulo_acc = ""

            h = acc.find(["h3", "h4"])

            if h:
                titulo_acc = limpiar_texto(
                    h.get_text()
                )

            texto = limpiar_texto(
                acc.get_text(" ")
            )

            # -----------------------------
            # Enlaces del acordeón
            # -----------------------------

            enlaces = []

            vistos = set()

            for a in acc.find_all(
                "a",
                href=True
            ):

                href = normalizar_url(
                    urljoin(
                        url,
                        a["href"]
                    )
                )

                if href in vistos:
                    continue

                vistos.add(href)

                enlaces.append({

                    "texto": limpiar_texto(
                        a.get_text(" ")
                    ),

                    "url": href

                })

            # -----------------------------
            # Banners del acordeón
            # -----------------------------

            banners = []

            for banner in acc.select(".banner"):

                titulo_banner = ""
                descripcion_banner = ""
                enlace_banner = None

                h = banner.find(["h3", "h4"])

                if h:
                    titulo_banner = limpiar_texto(
                        h.get_text()
                    )

                p = banner.find("p")

                if p:
                    descripcion_banner = limpiar_texto(
                        p.get_text(" ")
                    )

                a = banner.find(
                    "a",
                    href=True
                )

                if a:

                    enlace_banner = normalizar_url(
                        urljoin(
                            url,
                            a["href"]
                        )
                    )

                banners.append({

                    "titulo": titulo_banner,
                    "descripcion": descripcion_banner,
                    "url": enlace_banner

                })

            datos_seccion["acordeones"].append({

                "titulo": titulo_acc,
                "texto": texto,
                "enlaces": enlaces,
                "banners": banners

            })

        pagina["secciones"].append(
            datos_seccion
        )

    return pagina

In [ ]:
# ==========================================================
# BLOQUE 7. Generar JSON
# ==========================================================

padres = []

for nombre_json, url in FUENTES:

    print()
    print("=" * 70)
    print(url)
    print("=" * 70)

    pagina = extraer_padre(url)

    if pagina is not None:
        padres.append(pagina)


datos = {

    "fuente": "https://www.upv.es/admision/",

    "total_padres": len(padres),

    "padres": padres

}


ruta = os.path.join(

    CARPETA_JSON,

    "admision_grado.json"

)


with open(

    ruta,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        datos,

        f,

        ensure_ascii=False,

        indent=2

    )


print()
print("=" * 70)
print("Proceso terminado")
print("=" * 70)
print()

print(f"Padres: {len(padres)}")
print(f"JSON guardado en:\n{ruta}")


https://www.upv.es/admision/admision-grado/bachillerato-es.html
https://www.upv.es/admision/admision-grado/bachillerato-es.html

https://www.upv.es/admision/admision-grado/ciclos-formativos-es.html
https://www.upv.es/admision/admision-grado/ciclos-formativos-es.html

https://www.upv.es/admision/admision-grado/titulados-universitarios-es.html
https://www.upv.es/admision/admision-grado/titulados-universitarios-es.html

https://www.upv.es/admision/admision-grado/mayores-25-40-45-es.html
https://www.upv.es/admision/admision-grado/mayores-25-40-45-es.html

https://www.upv.es/admision/admision-grado/vengo-de-otra-universidad-es.html
https://www.upv.es/admision/admision-grado/vengo-de-otra-universidad-es.html

Proceso terminado

Padres: 5
JSON guardado en:
/content/drive/MyDrive/TFG Teleco/JSONs/admision_grado.json


In [ ]:
# ==========================================================
# COMPROBACIÓN DEL JSON
# ==========================================================

import json

with open(
    "/content/drive/MyDrive/TFG Teleco/JSONs/admision_grado.json",
    encoding="utf-8"
) as f:

    datos = json.load(f)

print("Padres:", len(datos["padres"]))

for padre in datos["padres"]:

    print()
    print("=" * 60)
    print(padre["titulo"])

    print(
        "Secciones:",
        len(padre["secciones"])
    )

    for s in padre["secciones"]:

        n_tarjetas = len(
            s.get("tarjetas", [])
        )

        n_acordeones = len(
            s.get("acordeones", [])
        )

        n_banners = len(
            s.get("banners", [])
        )

        n_enlaces = len(
            s.get("enlaces", [])
        )

        n_enlaces_acordeones = sum(

            len(
                acordeon.get("enlaces", [])
            )

            for acordeon in s.get(
                "acordeones",
                []
            )

        )

        total = (

            n_tarjetas
            + n_acordeones
            + n_banners
            + n_enlaces

        )

        print(
            f"{s['id']} - {s['titulo']}"
        )

        print(
            f"   Total: {total}"
        )

        print(
            f"      Tarjetas            : {n_tarjetas}"
        )

        print(
            f"      Acordeones          : {n_acordeones}"
        )

        print(
            f"      Banners             : {n_banners}"
        )

        print(
            f"      Enlaces sueltos     : {n_enlaces}"
        )

        print(
            f"      Links acordeones    : {n_enlaces_acordeones}"
        )

Padres: 5

Admisión grado - bachillerato | UPV - Universitat Politècnica de València
Secciones: 6
section-01 - Conoce la UPV
   Total: 5
      Tarjetas            : 5
      Acordeones          : 0
      Banners             : 0
      Enlaces sueltos     : 0
      Links acordeones    : 0
section-02 - Elige estudios
   Total: 6
      Tarjetas            : 6
      Acordeones          : 0
      Banners             : 0
      Enlaces sueltos     : 0
      Links acordeones    : 0
section-03 - Examínate de las PAU
   Total: 8
      Tarjetas            : 4
      Acordeones          : 4
      Banners             : 0
      Enlaces sueltos     : 0
      Links acordeones    : 4
section-04 - Haz tu preinscripción
   Total: 4
      Tarjetas            : 0
      Acordeones          : 4
      Banners             : 0
      Enlaces sueltos     : 0
      Links acordeones    : 10
section-05 - Realiza la matrícula
   Total: 2
      Tarjetas            : 0
      Acordeones          : 2
      Banners          

In [ ]:
#COMPROBACION DE UNA TARJETA
import pprint

pp = pprint.PrettyPrinter(
    width=120
)

pp.pprint(
    datos["padres"][4]["secciones"][2]
)

{'acordeones': [{'enlaces': [],
                 'texto': 'Para acceder a la solicitud, los miembros de la comunidad universitaria de la UPV deben '
                          'identificarse con su DNI y su PIN, mientras que los demás deben solicitar un identificador '
                          'previo a través del mismo enlace de la solicitud. Al realizar la solicitud de admisión, es '
                          'obligatorio cumplimentar también la solicitud de reconocimiento de créditos de las '
                          'asignaturas cursadas a través del apartado Gestión de reconocimiento.',
                 'titulo': ''}],
 'banners': [{'descripcion': 'El proceso de admisión a la UPV se realiza a través de la aplicación que la universidad '
                             'habilita para ello.',
              'titulo': 'Solicitud de admisión para continuar estudios',
              'url': 'https://www.upv.es/pls/soalu/sic_cont_est.sol_cont_est'},
             {'descripcion': '',
         

In [ ]:
# ==========================================================
# BLOQUE 8. Generación Markdown base para RAG
# Admisión grado + metadatos YAML
# ==========================================================

import json
import os
import re


# ----------------------------------------------------------
# Rutas
# ----------------------------------------------------------

ruta_json = os.path.join(
    CARPETA_JSON,
    "admision_grado.json"
)


directorio_base = os.path.join(
    ruta_programa,
    "ADMISION",
    "Grado"
)


os.makedirs(
    directorio_base,
    exist_ok=True
)


# ----------------------------------------------------------
# Parámetros
# ----------------------------------------------------------

CATEGORIA = "admision"
NIVEL = "grado"

INTRO_DOCUMENTO = (
    "Información completa sobre el proceso de admisión "
    "a estudios oficiales de grado en la "
    "Universitat Politècnica de València."
)


# ----------------------------------------------------------
# Limpieza de nombres de archivo
# ----------------------------------------------------------

def limpiar_nombre(nombre):

    nombre = nombre.lower()


    cambios = {

        "á": "a",
        "é": "e",
        "í": "i",
        "ó": "o",
        "ú": "u",
        "ü": "u",
        "ñ": "n"

    }


    for viejo, nuevo in cambios.items():

        nombre = nombre.replace(
            viejo,
            nuevo
        )


    nombre = re.sub(
        r"[^a-z0-9]+",
        "_",
        nombre
    )


    return nombre.strip("_")


# ----------------------------------------------------------
# Metadatos YAML
# ----------------------------------------------------------

def escribir_metadatos(
    f,
    tipo_documento,
    seccion=None
):

    f.write(
        "---\n"
    )

    f.write(
        f"fuente: UPV\n"
    )

    f.write(
        f"categoria: {CATEGORIA}\n"
    )

    f.write(
        f"nivel: {NIVEL}\n"
    )

    f.write(
        f"tipo_documento: {tipo_documento}\n"
    )

    if seccion:

        f.write(
            f"seccion: {seccion}\n"
        )

    f.write(
        "---\n\n"
    )


# ----------------------------------------------------------
# Cargar JSON
# ----------------------------------------------------------

with open(
    ruta_json,
    encoding="utf-8"
) as f:

    datos = json.load(f)


contador = 0


# ==========================================================
# GENERACIÓN
# ==========================================================

for padre in datos["padres"]:


    nombre_padre = limpiar_nombre(
        padre["titulo"]
    )


    # ------------------------------------------------------
    # Carpeta del padre
    # ------------------------------------------------------

    carpeta_padre = os.path.join(
        directorio_base,
        nombre_padre
    )


    os.makedirs(
        carpeta_padre,
        exist_ok=True
    )


    # ======================================================
    # DOCUMENTO PADRE
    # ======================================================

    archivo_padre = os.path.join(

        directorio_base,

        f"{nombre_padre}.md"

    )


    with open(
        archivo_padre,
        "w",
        encoding="utf-8"
    ) as f:


        escribir_metadatos(
            f,
            "padre"
        )


        f.write(
            f"# {padre['titulo']}\n\n"
        )


        f.write(
            INTRO_DOCUMENTO
            +
            "\n\n"
        )


        # --------------------------------------------------
        # Secciones
        # --------------------------------------------------

        for seccion in padre.get(
            "secciones",
            []
        ):


            f.write(
                f"## {seccion['titulo']}\n\n"
            )


            if seccion.get(
                "descripcion"
            ):

                f.write(
                    seccion["descripcion"]
                    +
                    "\n\n"
                )


            # ----------------------------------------------
            # Tarjetas
            # ----------------------------------------------

            for tarjeta in seccion.get(
                "tarjetas",
                []
            ):


                if tarjeta.get(
                    "titulo"
                ):

                    f.write(
                        f"### {tarjeta['titulo']}\n\n"
                    )


                if tarjeta.get(
                    "descripcion"
                ):

                    f.write(
                        tarjeta["descripcion"]
                        +
                        "\n\n"
                    )


                if tarjeta.get(
                    "url"
                ):

                    f.write(
                        f"Más información: "
                        f"{tarjeta['url']}\n\n"
                    )


            # ----------------------------------------------
            # Acordeones
            # ----------------------------------------------

            for acordeon in seccion.get(
                "acordeones",
                []
            ):


                if acordeon.get(
                    "titulo"
                ):

                    f.write(
                        f"### {acordeon['titulo']}\n\n"
                    )


                if acordeon.get(
                    "texto"
                ):

                    f.write(
                        acordeon["texto"]
                        +
                        "\n\n"
                    )


                # Enlaces del acordeón

                for enlace in acordeon.get(
                    "enlaces",
                    []
                ):

                    if (
                        enlace.get("texto")
                        and enlace.get("url")
                    ):

                        f.write(
                            f"- {enlace['texto']}: "
                            f"{enlace['url']}\n"
                        )


                if acordeon.get(
                    "enlaces"
                ):

                    f.write(
                        "\n"
                    )


                # Banners incluidos en acordeones

                for banner in acordeon.get(
                    "banners",
                    []
                ):


                    if banner.get(
                        "titulo"
                    ):

                        f.write(
                            f"#### {banner['titulo']}\n\n"
                        )


                    if banner.get(
                        "descripcion"
                    ):

                        f.write(
                            banner["descripcion"]
                            +
                            "\n\n"
                        )


                    if banner.get(
                        "url"
                    ):

                        f.write(
                            f"Más información: "
                            f"{banner['url']}\n\n"
                        )


    contador += 1


    # ======================================================
    # DOCUMENTOS POR SECCIÓN
    # ======================================================

    for seccion in padre.get(
        "secciones",
        []
    ):


        nombre_seccion = limpiar_nombre(
            seccion["titulo"]
        )


        archivo_seccion = os.path.join(

            carpeta_padre,

            f"{nombre_seccion}.md"

        )


        with open(
            archivo_seccion,
            "w",
            encoding="utf-8"
        ) as f:


            escribir_metadatos(
                f,
                "seccion",
                nombre_seccion
            )


            f.write(
                f"# {seccion['titulo']}\n\n"
            )


            f.write(
                f"Proceso de admisión: "
                f"{padre['titulo']}\n\n"
            )


            if seccion.get(
                "descripcion"
            ):

                f.write(
                    seccion["descripcion"]
                    +
                    "\n\n"
                )


            # ----------------------------------------------
            # Tarjetas
            # ----------------------------------------------

            for tarjeta in seccion.get(
                "tarjetas",
                []
            ):


                if tarjeta.get(
                    "titulo"
                ):

                    f.write(
                        f"## {tarjeta['titulo']}\n\n"
                    )


                if tarjeta.get(
                    "descripcion"
                ):

                    f.write(
                        tarjeta["descripcion"]
                        +
                        "\n\n"
                    )


                if tarjeta.get(
                    "url"
                ):

                    f.write(
                        f"Enlace oficial: "
                        f"{tarjeta['url']}\n\n"
                    )


            # ----------------------------------------------
            # Acordeones
            # ----------------------------------------------

            for acordeon in seccion.get(
                "acordeones",
                []
            ):


                if acordeon.get(
                    "titulo"
                ):

                    f.write(
                        f"## {acordeon['titulo']}\n\n"
                    )


                if acordeon.get(
                    "texto"
                ):

                    f.write(
                        acordeon["texto"]
                        +
                        "\n\n"
                    )


                # Enlaces

                for enlace in acordeon.get(
                    "enlaces",
                    []
                ):

                    if (
                        enlace.get("texto")
                        and enlace.get("url")
                    ):

                        f.write(
                            f"- {enlace['texto']}: "
                            f"{enlace['url']}\n"
                        )


                if acordeon.get(
                    "enlaces"
                ):

                    f.write(
                        "\n"
                    )


                # Banners

                for banner in acordeon.get(
                    "banners",
                    []
                ):


                    if banner.get(
                        "titulo"
                    ):

                        f.write(
                            f"## {banner['titulo']}\n\n"
                        )


                    if banner.get(
                        "descripcion"
                    ):

                        f.write(
                            banner["descripcion"]
                            +
                            "\n\n"
                        )


                    if banner.get(
                        "url"
                    ):

                        f.write(
                            f"Enlace oficial: "
                            f"{banner['url']}\n\n"
                        )


        contador += 1


# ==========================================================
# RESULTADO
# ==========================================================

print()
print(
    "Markdown base generado correctamente."
)
print()
print(
    "Archivos creados:",
    contador
)
print()
print(
    "Ruta:",
    directorio_base
)


Markdown base generado correctamente.

Archivos creados: 33

Ruta: /content/drive/MyDrive/TFG Teleco/ADMISION/Grado


In [ ]:
# ==========================================================
# BLOQUE 9. Extracción de enlaces para generar recursos RAG
# Admisión Grado
# ==========================================================

import json
from urllib.parse import urljoin, urlparse, urlunparse


# ----------------------------------------------------------
# Rutas
# ----------------------------------------------------------

ruta_json = os.path.join(
    CARPETA_JSON,
    "admision_grado.json"
)


# ----------------------------------------------------------
# Normalización de URLs
# ----------------------------------------------------------

def normalizar_url_recurso(url):

    if not url:
        return ""

    p = urlparse(url)

    return urlunparse(
        (
            p.scheme,
            p.netloc,
            p.path,
            "",
            "",
            ""
        )
    )


# ----------------------------------------------------------
# Cargar JSON
# ----------------------------------------------------------

with open(
    ruta_json,
    encoding="utf-8"
) as f:

    datos = json.load(f)


# ----------------------------------------------------------
# Extracción
# ----------------------------------------------------------

lista_enlaces = []

vistos = set()


for padre in datos["padres"]:

    titulo_padre = padre["titulo"]


    for seccion in padre["secciones"]:

        titulo_seccion = seccion["titulo"]


        # ==================================================
        # TARJETAS
        # ==================================================

        for tarjeta in seccion.get(
            "tarjetas",
            []
        ):

            url = tarjeta.get(
                "url"
            )

            if not url:
                continue


            url = normalizar_url_recurso(
                url
            )


            if not url:
                continue


            clave = (
                titulo_padre,
                titulo_seccion,
                url
            )


            if clave in vistos:
                continue


            vistos.add(
                clave
            )


            lista_enlaces.append({

                "url": url,

                "texto": tarjeta.get(
                    "titulo",
                    ""
                ),

                "seccion_origen":
                    titulo_seccion,

                "padre_origen":
                    titulo_padre,

                "tipo_origen":
                    "tarjeta"

            })


        # ==================================================
        # ACORDEONES
        # ==================================================

        for acordeon in seccion.get(
            "acordeones",
            []
        ):


            # ----------------------------------------------
            # Enlaces del acordeón
            # ----------------------------------------------

            for enlace in acordeon.get(
                "enlaces",
                []
            ):

                url = enlace.get(
                    "url"
                )

                if not url:
                    continue


                url = normalizar_url_recurso(
                    url
                )


                if not url:
                    continue


                clave = (
                    titulo_padre,
                    titulo_seccion,
                    url
                )


                if clave in vistos:
                    continue


                vistos.add(
                    clave
                )


                lista_enlaces.append({

                    "url": url,

                    "texto": enlace.get(
                        "texto",
                        ""
                    ),

                    "seccion_origen":
                        titulo_seccion,

                    "padre_origen":
                        titulo_padre,

                    "tipo_origen":
                        "acordeon"

                })


            # ----------------------------------------------
            # Banners del acordeón
            # ----------------------------------------------

            for banner in acordeon.get(
                "banners",
                []
            ):

                url = banner.get(
                    "url"
                )

                if not url:
                    continue


                url = normalizar_url_recurso(
                    url
                )


                if not url:
                    continue


                clave = (
                    titulo_padre,
                    titulo_seccion,
                    url
                )


                if clave in vistos:
                    continue


                vistos.add(
                    clave
                )


                lista_enlaces.append({

                    "url": url,

                    "texto": banner.get(
                        "titulo",
                        ""
                    ),

                    "seccion_origen":
                        titulo_seccion,

                    "padre_origen":
                        titulo_padre,

                    "tipo_origen":
                        "banner"

                })


# ----------------------------------------------------------
# Resumen
# ----------------------------------------------------------

print()
print("=" * 70)
print("ENLACES PARA RECURSOS RAG")
print("=" * 70)

print()
print(
    "Enlaces encontrados:",
    len(lista_enlaces)
)

print()


for enlace in lista_enlaces:

    print(
        "-",
        enlace["seccion_origen"]
    )

    print(
        "  Tipo:",
        enlace["tipo_origen"]
    )

    print(
        "  Texto:",
        enlace["texto"]
    )

    print(
        "  URL:",
        enlace["url"]
    )

    print()


ENLACES PARA RECURSOS RAG

Enlaces encontrados: 125

- Conoce la UPV
  Tipo: tarjeta
  Texto: Jornadas de Puertas Abiertas
  URL: https://www.jpa.upv.es/

- Conoce la UPV
  Tipo: tarjeta
  Texto: Videopódcasts OPEN
  URL: https://www.upv.es/contenidos/jpa/sesiones-on-line-2/

- Conoce la UPV
  Tipo: tarjeta
  Texto: Red de Embajadores
  URL: https://www.upv.es/contenidos/embajadores/

- Conoce la UPV
  Tipo: tarjeta
  Texto: Campus Praktikum
  URL: https://www.upv.es/contenidos/praktikum/

- Conoce la UPV
  Tipo: tarjeta
  Texto: Geocaching
  URL: https://geocaching.upv.es/portada/cas/index.html

- Elige estudios
  Tipo: tarjeta
  Texto: IA orientadora
  URL: https://www.upv.es/contenidos/asistenteia/

- Elige estudios
  Tipo: tarjeta
  Texto: Grados y dobles grados
  URL: https://www.upv.es/estudios/grado/index-es.html

- Elige estudios
  Tipo: tarjeta
  Texto: Estudia en China
  URL: https://www.upv.es/china

- Elige estudios
  Tipo: tarjeta
  Texto: Dobles titulaciones internaciona

In [ ]:
# ==========================================================
# BLOQUE 10. Descarga, filtrado y extracción de recursos
#
# Admisión Grado
# ==========================================================

import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urlunparse

# ----------------------------------------------------------
# Configuración
# ----------------------------------------------------------

paginas_extraidas = []

# ----------------------------------------------------------
# Dominios UPV permitidos
#
# Se acepta cualquier recurso perteneciente al ecosistema
# principal de la UPV, sin restringir por rutas concretas.
#
# Esto permite aceptar, por ejemplo:
#   www.upv.es
#   upv.es
#   jpa.upv.es
#
# mientras que dominios externos como universitats.gva.es,
# zoom.us, videos.gva.es, etc. quedan fuera.
# ----------------------------------------------------------

DOMINIOS_UPV = {
    "www.upv.es",
    "upv.es"
}

# ----------------------------------------------------------
# Funciones auxiliares
# ----------------------------------------------------------

def normalizar_url_final(url):

    if not url:
        return None

    p = urlparse(url)

    return urlunparse(
        (
            p.scheme.lower(),
            p.netloc.lower(),
            p.path.rstrip("/") or "/",
            "",
            p.query,
            ""
        )
    )


def es_url_grado(url):

    """
    Determina si una URL pertenece al ecosistema UPV útil.

    Se acepta:
        - www.upv.es
        - upv.es
        - subdominios de upv.es

    No se restringe por rutas concretas, ya que páginas
    potencialmente útiles pueden encontrarse en distintas
    zonas del portal.
    """

    if not url:
        return False

    try:

        p = urlparse(url)

        dominio = p.netloc.lower().split(":")[0]

    except Exception:

        return False

    # Dominio principal
    if dominio in DOMINIOS_UPV:
        return True

    # Subdominios de upv.es
    if dominio.endswith(".upv.es"):
        return True

    return False


def eliminar_basura(soup):

    for elemento in soup.find_all(
        [
            "script",
            "style",
            "noscript",

            "header",
            "footer",
            "nav",

            "aside",

            "form",
            "iframe"
        ]
    ):

        elemento.decompose()


def encontrar_contenido_principal(soup):

    """
    Busca contenido principal sin exigir una única estructura
    HTML concreta de la UPV.
    """

    selectores = [
        "main",
        "article",
        "#content",
        ".content",
        ".container",
        ".main-content"
    ]

    for selector in selectores:

        elemento = soup.select_one(selector)

        if elemento is not None:

            texto = elemento.get_text(
                " ",
                strip=True
            )

            if len(texto) >= 100:

                return elemento

    # ------------------------------------------------------
    # Algunas páginas de la UPV no utilizan ninguno de los
    # selectores anteriores.
    #
    # Como último recurso se utiliza el body si contiene
    # suficiente texto.
    # ------------------------------------------------------

    body = soup.find("body")

    if body is not None:

        texto = body.get_text(
            " ",
            strip=True
        )

        if len(texto) >= 100:

            return body

    return None


def normalizar_texto_para_comparacion(texto):

    return " ".join(
        texto.lower().split()
    )


def contenido_claramente_ajeno(
    titulo,
    contenido,
    url
):

    """
    Filtro deliberadamente conservador.

    Solo descarta contenido cuando existen varias señales
    claras de que la página pertenece a otro ámbito.

    No se utiliza una clasificación semántica agresiva.
    """

    texto = (
        titulo
        + " "
        + contenido
        + " "
        + url
    ).lower()

    # ------------------------------------------------------
    # Indicadores inequívocos de contenido ajeno
    # ------------------------------------------------------

    patrones_ajenos = [

        # Doctorado / investigación
        "doctorado",
        "tesis doctoral",
        "doctorando",

        # Personal universitario
        "personal docente e investigador",
        "personal investigador",
        "pdi",
        "profesorado",

        # Investigación
        "proyecto de investigación",
        "proyectos de investigación",
        "grupo de investigación",
        "grupos de investigación",

        # Empresas / transferencia
        "empresa de base tecnológica",
        "spin-off",
        "transferencia tecnológica",

        # Alumni
        "antiguos alumnos",
        "exalumnos",

    ]

    coincidencias = sum(
        1
        for patron in patrones_ajenos
        if patron in texto
    )

    # ------------------------------------------------------
    # Solo se descarta con varias señales simultáneas.
    # ------------------------------------------------------

    if coincidencias >= 2:

        return True

    return False


def contenido_demasiado_corto(contenido):

    return len(
        contenido.strip()
    ) < 100


# ==========================================================
# Deduplicación previa
# ==========================================================

urls_entrada_vistas = set()

enlaces_unicos = []

for enlace in lista_enlaces:

    url_original = enlace.get(
        "url",
        ""
    )

    if not url_original:
        continue

    url_normalizada = normalizar_url_final(
        url_original
    )

    if not url_normalizada:
        continue

    if url_normalizada in urls_entrada_vistas:
        continue

    urls_entrada_vistas.add(
        url_normalizada
    )

    enlaces_unicos.append(
        enlace
    )


print()

print(
    "Enlaces originales:",
    len(lista_enlaces)
)

print(
    "Enlaces únicos antes de descargar:",
    len(enlaces_unicos)
)


# ==========================================================
# Deduplicación de URLs finales
#
# Dos URLs distintas pueden redirigir a la misma página.
# ==========================================================

vistos_urls_finales = set()


# ==========================================================
# Sesión HTTP
# ==========================================================

sesion = requests.Session()

sesion.headers.update(
    HEADERS
)


# ==========================================================
# Descarga
# ==========================================================

for enlace in enlaces_unicos:

    url_original = enlace["url"]

    print()
    print("=" * 80)
    print(url_original)


    # ======================================================
    # Comprobar ámbito antes de descargar
    # ======================================================

    if not es_url_grado(
        url_original
    ):

        print(
            "Descartado (URL fuera del ámbito útil)"
        )

        continue


    # ======================================================
    # Descargar
    # ======================================================

    try:

        respuesta = sesion.get(

            url_original,

            timeout=20,

            allow_redirects=True

        )

        respuesta.raise_for_status()


    except Exception as e:

        print(
            "Error de descarga:",
            e
        )

        continue


    # ------------------------------------------------------
    # URL final después de redirecciones
    # ------------------------------------------------------

    url_final = normalizar_url_final(
        respuesta.url
    )


    print(
        "URL final:",
        url_final
    )


    # ------------------------------------------------------
    # Comprobar ámbito después de redirección
    # ------------------------------------------------------

    if not es_url_grado(
        url_final
    ):

        print(
            "Descartado (URL final no útil)"
        )

        continue


    # ------------------------------------------------------
    # Evitar duplicados después de redirección
    # ------------------------------------------------------

    if url_final in vistos_urls_finales:

        print(
            "Descartado (URL final duplicada)"
        )

        continue


    vistos_urls_finales.add(
        url_final
    )


    # ------------------------------------------------------
    # Comprobar tipo de contenido
    # ------------------------------------------------------

    content_type = respuesta.headers.get(
        "Content-Type",
        ""
    ).lower()


    # ------------------------------------------------------
    # Este bloque trabaja con HTML.
    #
    # PDFs, vídeos, imágenes, etc. se descartan aquí porque
    # requieren extractores específicos.
    # ------------------------------------------------------

    if "text/html" not in content_type:

        print(
            "Descartado (recurso no HTML)"
        )

        continue


    # ======================================================
    # Parsear HTML
    # ======================================================

    soup = BeautifulSoup(

        respuesta.text,

        "html.parser"

    )


    eliminar_basura(
        soup
    )


    # ======================================================
    # Buscar contenido principal
    # ======================================================

    contenido_principal = encontrar_contenido_principal(
        soup
    )


    if contenido_principal is None:

        print(
            "Descartado (no se encontró contenido principal)"
        )

        continue


    # ======================================================
    # Título
    # ======================================================

    titulo = ""


    h1 = contenido_principal.find(
        "h1"
    )


    if h1:

        titulo = limpiar_texto(
            h1.get_text(" ")
        )


    if not titulo:

        h1 = soup.find(
            "h1"
        )

        if h1:

            titulo = limpiar_texto(
                h1.get_text(" ")
            )


    if not titulo:

        titulo = enlace.get(
            "texto",
            ""
        )


    titulo = limpiar_texto(
        titulo
    )


    # ======================================================
    # Extracción estructurada
    # ======================================================

    bloques = []


    for elemento in contenido_principal.find_all(

        [
            "h1",
            "h2",
            "h3",
            "h4",
            "p",
            "li",
            "table"
        ]

    ):

        texto = limpiar_texto(

            elemento.get_text(
                " ",
                strip=True
            )

        )


        if len(texto) < 3:

            continue


        # --------------------------------------------------
        # Evitar repetir el título principal
        # --------------------------------------------------

        if (

            elemento.name == "h1"

            and normalizar_texto_para_comparacion(
                texto
            )
            ==
            normalizar_texto_para_comparacion(
                titulo
            )

        ):

            continue


        bloques.append(
            texto
        )


    # ======================================================
    # Eliminar duplicados consecutivos
    # ======================================================

    bloques_limpios = []


    for bloque in bloques:

        if (

            bloques_limpios

            and normalizar_texto_para_comparacion(
                bloque
            )
            ==
            normalizar_texto_para_comparacion(
                bloques_limpios[-1]
            )

        ):

            continue


        bloques_limpios.append(
            bloque
        )


    contenido = "\n\n".join(
        bloques_limpios
    )


    # ======================================================
    # Filtrado de contenido insuficiente
    # ======================================================

    if contenido_demasiado_corto(
        contenido
    ):

        print(
            "Descartado (contenido insuficiente)"
        )

        continue


    # ======================================================
    # Filtrado de contenido claramente ajeno
    #
    # Se mantiene deliberadamente conservador.
    # ======================================================

    if contenido_claramente_ajeno(

        titulo,

        contenido,

        url_final

    ):

        print(
            "Descartado (contenido claramente ajeno)"
        )

        continue


    # ======================================================
    # Guardar resultado
    # ======================================================

    paginas_extraidas.append({

        "url_original":
            url_original,

        "url":
            url_final,

        "titulo":
            titulo,

        "texto_enlace":
            enlace.get(
                "texto",
                ""
            ),

        "seccion_origen":
            enlace.get(
                "seccion_origen",
                ""
            ),

        "padre_origen":
            enlace.get(
                "padre_origen",
                ""
            ),

        "tipo_origen":
            enlace.get(
                "tipo_origen",
                ""
            ),

        "contenido":
            contenido

    })


    print(
        "Página aceptada"
    )


# ==========================================================
# Resumen
# ==========================================================

print()
print("=" * 80)
print("RESULTADO DE LA EXTRACCIÓN")
print("=" * 80)
print()


print(
    "Enlaces originales:",
    len(lista_enlaces)
)


print(
    "Enlaces únicos procesados:",
    len(enlaces_unicos)
)


print(
    "URLs finales únicas:",
    len(vistos_urls_finales)
)


print(
    "Páginas útiles:",
    len(paginas_extraidas)
)


print()


# ==========================================================
# Mostrar primeras páginas
# ==========================================================

for pagina in paginas_extraidas[:5]:

    print("=" * 80)

    print(
        pagina["titulo"]
    )

    print()

    print(
        "Sección:",
        pagina["seccion_origen"]
    )

    print(
        "URL:",
        pagina["url"]
    )

    print()

    print(
        pagina["contenido"][:1000]
    )

    print()


Enlaces originales: 125
Enlaces únicos antes de descargar: 36

https://www.jpa.upv.es/
URL final: https://www.upv.es/contenidos/jpa
Página aceptada

https://www.upv.es/contenidos/jpa/sesiones-on-line-2/
URL final: https://www.upv.es/contenidos/jpa/sesiones-on-line-2
Página aceptada

https://www.upv.es/contenidos/embajadores/
URL final: https://www.upv.es/contenidos/embajadores
Página aceptada

https://www.upv.es/contenidos/praktikum/
URL final: https://www.upv.es/contenidos/praktikum
Página aceptada

https://geocaching.upv.es/portada/cas/index.html
Error de descarga: HTTPSConnectionPool(host='geocaching.upv.es', port=443): Max retries exceeded with url: /portada/cas/index.html (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7e5832ae7c80>, 'Connection to geocaching.upv.es timed out. (connect timeout=20)'))

https://www.upv.es/contenidos/asistenteia/
URL final: https://www.upv.es/contenidos/asistenteia
Página aceptada

https://www.upv.es/estudios/grado/ind

In [ ]:
# ==========================================================
# BLOQUE 12. Generación de Markdown de páginas enlazadas
#
# Admisión Grado + recursos por padre
# ==========================================================

import os
import re

# ----------------------------------------------------------
# Configuración
# ----------------------------------------------------------

directorio_base = os.path.join(
    ruta_programa,
    "ADMISION",
    "Grado"
)

CATEGORIA = "admision"
NIVEL = "grado"

# ----------------------------------------------------------
# Limpieza de nombres de archivo y carpetas
# ----------------------------------------------------------

def limpiar_nombre(nombre):

    nombre = str(nombre).lower().strip()

    cambios = {
        "á": "a",
        "é": "e",
        "í": "i",
        "ó": "o",
        "ú": "u",
        "ü": "u",
        "ñ": "n"
    }

    for viejo, nuevo in cambios.items():

        nombre = nombre.replace(
            viejo,
            nuevo
        )

    nombre = re.sub(
        r"[^a-z0-9]+",
        "_",
        nombre
    )

    return nombre.strip("_")


# ==========================================================
# Clasificación del recurso
# ==========================================================

def clasificar_recurso(pagina):

    texto = (
        pagina.get("titulo", "")
        + " "
        + pagina.get("url", "")
        + " "
        + pagina.get("seccion_origen", "")
    ).lower()

    # ------------------------------------------------------
    # Plazos / calendario
    # ------------------------------------------------------

    if any(
        palabra in texto
        for palabra in [
            "plazo",
            "calendario",
            "calendarios",
            "fechas"
        ]
    ):

        return "calendario"


    # ------------------------------------------------------
    # Matrícula
    # ------------------------------------------------------

    if any(
        palabra in texto
        for palabra in [
            "matricula",
            "matrícula",
            "tasas",
            "precio",
            "precios"
        ]
    ):

        return "matricula"


    # ------------------------------------------------------
    # Preguntas frecuentes
    # ------------------------------------------------------

    if any(
        palabra in texto
        for palabra in [
            "faq",
            "faqs",
            "preguntas frecuentes"
        ]
    ):

        return "faq"


    # ------------------------------------------------------
    # Becas / ayudas
    # ------------------------------------------------------

    if any(
        palabra in texto
        for palabra in [
            "beca",
            "becas",
            "ayuda",
            "ayudas"
        ]
    ):

        return "ayudas"


    # ------------------------------------------------------
    # Admisión / preinscripción
    # ------------------------------------------------------

    if any(
        palabra in texto
        for palabra in [
            "admision",
            "admisión",
            "preinscripcion",
            "preinscripción",
            "solicitud",
            "acceso"
        ]
    ):

        return "admision"


    # ------------------------------------------------------
    # Estudios / titulaciones
    # ------------------------------------------------------

    if any(
        palabra in texto
        for palabra in [
            "grado",
            "grados",
            "estudio",
            "estudios",
            "titulacion",
            "titulación",
            "titulaciones",
            "oferta academica",
            "oferta académica"
        ]
    ):

        return "estudios"


    # ------------------------------------------------------
    # Normativa
    # ------------------------------------------------------

    if any(
        palabra in texto
        for palabra in [
            "normativa",
            "reglamento",
            "legislacion",
            "legislación"
        ]
    ):

        return "normativa"


    # ------------------------------------------------------
    # Por defecto
    # ------------------------------------------------------

    return "informacion"


# ==========================================================
# Metadatos YAML
# ==========================================================

def escribir_metadatos(
    f,
    pagina,
    tipo_recurso
):

    f.write(
        "---\n"
    )

    f.write(
        "fuente: UPV\n"
    )

    f.write(
        f"categoria: {CATEGORIA}\n"
    )

    f.write(
        f"nivel: {NIVEL}\n"
    )

    f.write(
        "tipo_documento: recurso\n"
    )

    f.write(
        f"tipo_recurso: {tipo_recurso}\n"
    )

    f.write(
        "padre: "
        + limpiar_nombre(
            pagina.get(
                "padre_origen",
                ""
            )
        )
        + "\n"
    )

    f.write(
        "seccion: "
        + limpiar_nombre(
            pagina.get(
                "seccion_origen",
                ""
            )
        )
        + "\n"
    )

    f.write(
        f"url: {pagina['url']}\n"
    )

    f.write(
        "---\n\n"
    )


# ==========================================================
# DEDUPLICACIÓN DE PÁGINAS
#
# Una misma URL puede aparecer varias veces porque diferentes
# enlaces del JSON apuntan al mismo recurso.
# ==========================================================

paginas_unicas = []

urls_vistas = set()

for pagina in paginas_extraidas:

    url = pagina.get(
        "url",
        ""
    )

    if not url:
        continue


    if url in urls_vistas:

        continue


    urls_vistas.add(
        url
    )

    paginas_unicas.append(
        pagina
    )


# ==========================================================
# GENERACIÓN DE RECURSOS
# ==========================================================

contador = 0

for pagina in paginas_unicas:

    # ------------------------------------------------------
    # Obtener padre y sección de origen
    # ------------------------------------------------------

    padre_origen = pagina.get(
        "padre_origen",
        ""
    )

    seccion_origen = pagina.get(
        "seccion_origen",
        ""
    )


    # ------------------------------------------------------
    # Si por algún motivo no existe padre de origen,
    # no podemos determinar correctamente dónde colocar
    # el recurso.
    # ------------------------------------------------------

    if not padre_origen:

        print(
            "Recurso omitido: "
            "no se encontró padre_origen"
        )

        print(
            pagina.get(
                "url",
                ""
            )
        )

        continue


    # ------------------------------------------------------
    # Nombre normalizado del padre
    # ------------------------------------------------------

    nombre_padre = limpiar_nombre(
        padre_origen
    )


    if not nombre_padre:

        nombre_padre = "padre"


    # ------------------------------------------------------
    # Carpeta del padre
    # ------------------------------------------------------

    carpeta_padre = os.path.join(

        directorio_base,

        nombre_padre

    )


    os.makedirs(

        carpeta_padre,

        exist_ok=True

    )


    # ------------------------------------------------------
    # Carpeta de recursos
    # ------------------------------------------------------

    directorio_recursos = os.path.join(

        carpeta_padre,

        "recursos"

    )


    os.makedirs(

        directorio_recursos,

        exist_ok=True

    )


    # ------------------------------------------------------
    # Clasificación
    # ------------------------------------------------------

    tipo_recurso = clasificar_recurso(
        pagina
    )


    # ------------------------------------------------------
    # Nombre del recurso
    # ------------------------------------------------------

    nombre = limpiar_nombre(

        pagina.get(
            "titulo",
            ""
        )

    )


    if not nombre:

        nombre = "recurso"


    # ------------------------------------------------------
    # Nombre de archivo inicial
    # ------------------------------------------------------

    archivo = os.path.join(

        directorio_recursos,

        f"{nombre}.md"

    )


    # ------------------------------------------------------
    # Evitar colisiones de nombres
    # ------------------------------------------------------

    if os.path.exists(
        archivo
    ):

        nombre_seccion = limpiar_nombre(

            seccion_origen

        )


        if nombre_seccion:

            archivo = os.path.join(

                directorio_recursos,

                f"{nombre}_{nombre_seccion}.md"

            )


    # ------------------------------------------------------
    # Si continúa existiendo, añadir contador
    # ------------------------------------------------------

    contador_nombre = 2

    archivo_base = archivo

    nombre_archivo, extension = os.path.splitext(
        archivo_base
    )


    while os.path.exists(
        archivo
    ):

        archivo = os.path.join(

            directorio_recursos,

            f"{nombre_archivo}_{contador_nombre}{extension}"

        )

        contador_nombre += 1


    # ======================================================
    # ESCRITURA DEL MARKDOWN
    # ======================================================

    with open(

        archivo,

        "w",

        encoding="utf-8"

    ) as f:

        # --------------------------------------------------
        # YAML
        # --------------------------------------------------

        escribir_metadatos(

            f,

            pagina,

            tipo_recurso

        )


        # --------------------------------------------------
        # Título
        # --------------------------------------------------

        titulo = pagina.get(
            "titulo",
            "Recurso UPV"
        )


        f.write(
            f"# {titulo}\n\n"
        )


        # --------------------------------------------------
        # Contexto
        # --------------------------------------------------

        f.write(
            "Recurso relacionado con el proceso de "
            "admisión a estudios oficiales de grado "
            "en la Universitat Politècnica de València.\n\n"
        )


        # --------------------------------------------------
        # Contexto de procedencia
        # --------------------------------------------------

        if padre_origen:

            f.write(
                f"Proceso de admisión: "
                f"{padre_origen}\n\n"
            )


        if seccion_origen:

            f.write(
                f"Sección de origen: "
                f"{seccion_origen}\n\n"
            )


        # --------------------------------------------------
        # Contenido extraído
        # --------------------------------------------------

        contenido = pagina.get(
            "contenido",
            ""
        )


        if contenido:

            f.write(
                contenido
                +
                "\n\n"
            )


        # --------------------------------------------------
        # Fuente oficial
        # --------------------------------------------------

        f.write(
            f"Fuente oficial: "
            f"{pagina['url']}\n"
        )


    contador += 1


    # ------------------------------------------------------
    # Información de depuración
    # ------------------------------------------------------

    print(
        "Recurso generado:"
    )

    print(
        f"  Padre   : {padre_origen}"
    )

    print(
        f"  Sección : {seccion_origen}"
    )

    print(
        f"  Tipo    : {tipo_recurso}"
    )

    print(
        f"  Archivo : {archivo}"
    )

    print()


# ==========================================================
# RESULTADO
# ==========================================================

print()
print("=" * 70)
print(
    "MARKDOWN DE RECURSOS GENERADO CORRECTAMENTE"
)
print("=" * 70)
print()

print(
    "Enlaces originales:",
    len(lista_enlaces)
)

print(
    "Páginas descargadas:",
    len(paginas_extraidas)
)

print(
    "Páginas únicas:",
    len(paginas_unicas)
)

print(
    "Archivos creados:",
    contador
)

print()

print(
    "Ruta:",
    directorio_base
)

Recurso generado:
  Padre   : Admisión grado - bachillerato | UPV - Universitat Politècnica de València
  Sección : Conoce la UPV
  Tipo    : informacion
  Archivo : /content/drive/MyDrive/TFG Teleco/ADMISION/Grado/admision_grado_bachillerato_upv_universitat_polit_cnica_de_val_ncia/recursos/jornadas_de_puertas_abiertas_upv.md

Recurso generado:
  Padre   : Admisión grado - bachillerato | UPV - Universitat Politècnica de València
  Sección : Conoce la UPV
  Tipo    : informacion
  Archivo : /content/drive/MyDrive/TFG Teleco/ADMISION/Grado/admision_grado_bachillerato_upv_universitat_polit_cnica_de_val_ncia/recursos/videopodcasts_open.md

Recurso generado:
  Padre   : Admisión grado - bachillerato | UPV - Universitat Politècnica de València
  Sección : Conoce la UPV
  Tipo    : informacion
  Archivo : /content/drive/MyDrive/TFG Teleco/ADMISION/Grado/admision_grado_bachillerato_upv_universitat_polit_cnica_de_val_ncia/recursos/red_de_embajadores.md

Recurso generado:
  Padre   : Admisión gr